# Earth Engine + fused datacube: LULC + Sentinel-2 + DEM

<a href="https://colab.research.google.com/github/buckai-observatory/geoai-datacubes/blob/feature/earth-engine-provider/notebooks/04_earth_engine_dynamic_world.ipynb" target="_blank" rel="noopener noreferrer"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>
<!-- BRANCH-PREVIEW: swap `feature/earth-engine-provider` -> `main` at merge time -->


Fetches a **Dynamic World** (Google Earth Engine, per-Sentinel-2-scene 9-class
LULC) **OR ESA-WorldCover** (static 2020/2021 10 m LULC) map, a **Sentinel-2**
scene (RGB + NIR), and a **Copernicus DEM** tile over the same AOI; fuses them
into a single multi-band datacube on a common UTM grid; and trains a
lightweight XGBoost pixel classifier that predicts a target LULC class from
the S2 + DEM features. This is the "easy" onramp to the pipeline -- one Colab
click, one AOI, one classifier, live labels.

Two LULC branches share the same fetch / fuse / train code path:

- `LABEL_SOURCE = "Dynamic-World"` -- Earth Engine only; needs a time range;
  time-reduces N scenes via `mode` on the label band (`GOOGLE/DYNAMICWORLD/V1`).
- `LABEL_SOURCE = "ESA-WorldCover"` -- Planetary Computer STAC;
  static 2020/2021 mosaic; ignores `TIME_RANGE`.

Prerequisites:

- A Google account with Earth Engine access
  (<https://developers.google.com/earth-engine/guides/access>) if you want
  the `Dynamic-World` branch; ESA-WorldCover works without EE.
- The `earthengine` + `ml` install extras (the Colab cell below handles this):
  `pip install geoai-datacubes[earthengine,ml]`.

See [`docs/providers/earth_engine.md`](../docs/providers/earth_engine.md)
for the full auth / quota / tiling reference.


In [ ]:
# --- Colab / local bootstrap ---
# On Colab, this cell clones the repo, installs geoai-datacubes with the
# [earthengine,ml] extras, and kicks off EE's interactive OAuth flow (which
# pops a Google sign-in window in the Colab UI). On a local Jupyter run
# it is a no-op that just confirms the repo layout and assumes you have
# already authenticated with EE at least once (see docs/providers/earth_engine.md).
#
# BRANCH-PREVIEW: while feature/earth-engine-provider is still unmerged we
# clone that branch so the [earthengine] extra + Dynamic-World profile exist
# in the checkout. Swap `BRANCH` back to "main" at merge time.
BRANCH = "feature/earth-engine-provider"

import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print(f"Colab detected -- bootstrapping repo + dependencies (branch: {BRANCH})")
    REPO_DIR = Path("/content/geoai-datacubes")
    if not REPO_DIR.exists():
        subprocess.check_call([
            "git", "clone", "--depth", "1", "--branch", BRANCH,
            "https://github.com/buckai-observatory/geoai-datacubes.git",
            str(REPO_DIR),
        ])

    # Install the package with [earthengine,ml] so `import ee` + `import xgboost` work.
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        f"{REPO_DIR}[earthengine,ml]",
    ])

    os.chdir(REPO_DIR / "notebooks")
    print(f"cwd = {os.getcwd()}")

    # If a Colab secret named EARTHENGINE_TOKEN is present, use it and skip
    # the interactive OAuth flow entirely (recommended for repeated runs).
    try:
        from google.colab import userdata
        token = userdata.get("EARTHENGINE_TOKEN")
        if token:
            os.environ["EARTHENGINE_TOKEN"] = token
            print("Using EARTHENGINE_TOKEN from Colab userdata secrets.")
    except Exception:
        pass

    if not os.environ.get("EARTHENGINE_TOKEN"):
        import ee
        # Interactive: opens a Google sign-in popup and writes the resulting
        # credentials to ~/.config/earthengine/credentials.
        ee.Authenticate()
else:
    print("Local environment -- using existing checkout")
    print("Assuming EE is authenticated (see docs/providers/earth_engine.md)")


## Setup

Define the AOI, time window, and LULC-source toggle. The AOI is a
~5 km x ~5.5 km bbox over Columbus, OH -- deliberately small so the whole
notebook (four fetches + fuse + XGBoost train + eval) finishes on Colab in
under ten minutes. The mission's `PROVIDER_AUTO` routing dispatches
`Dynamic-World` to the `earth_engine` provider and `ESA-WorldCover` to
Planetary Computer, so `provider="auto"` works for both. Note that
ESA-WorldCover is a static 2020/2021 mosaic and ignores `TIME_RANGE`.


In [ ]:
import os
import sys
import json
import time
from pathlib import Path

import numpy as np
import rasterio
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm

# locate the repo root from this notebook's CWD (notebooks/)
NB_DIR = Path.cwd()
if NB_DIR.name != "notebooks":
    for p in (NB_DIR, *NB_DIR.parents):
        if (p / "notebooks").is_dir() and (p / "geoai_datacubes").is_dir():
            NB_DIR = p / "notebooks"
            break
REPO_ROOT = NB_DIR.parent
sys.path.insert(0, str(REPO_ROOT))

# Scratch folders for this notebook.
OUT  = NB_DIR / "_outputs_nb04"
DATA = OUT / "data"
for d in (OUT, DATA):
    d.mkdir(parents=True, exist_ok=True)

AOI          = [-83.06, 39.97, -83.00, 40.02]   # ~5 km x 5.5 km, Columbus OH
TIME_RANGE   = ("2024-07-01", "2024-07-31")     # ignored for ESA-WorldCover
RESOLUTION   = 10                                # metres

# User choice: "Dynamic-World" (live per-scene, needs TIME_RANGE) or
# "ESA-WorldCover" (static 2020/2021 mosaic, ignores TIME_RANGE).
LABEL_SOURCE = "Dynamic-World"

TARGET_NAME     = "built"                        # human-readable target class
# Class-id mapping for each LULC product's LULC band.
TARGET_CLASS_ID = {"Dynamic-World": 6, "ESA-WorldCover": 50}[LABEL_SOURCE]

print(f"AOI          : {AOI}")
print(f"Time range   : {TIME_RANGE}")
print(f"Resolution   : {RESOLUTION} m")
print(f"LULC source  : {LABEL_SOURCE}  (target class {TARGET_CLASS_ID} = {TARGET_NAME!r})")
print(f"Output dir   : {DATA}")


## Fetch the LULC label layer

One call, dispatched by `LABEL_SOURCE`. The pipeline picks the provider from
the mission registry:

- **Dynamic-World**: `earth_engine` provider. Reduces the ~4-8 scenes in
  `TIME_RANGE` server-side using each band's reducer from
  `MISSION_PROFILES["Dynamic-World"]["providers"]["earth_engine"]["reducer_groups"]`
  -- `mean` for the 9 probability bands, `mode` for the hard `LULC` label.
- **ESA-WorldCover**: `planetary_computer` STAC provider. Static 2020/2021
  mosaic; `TIME_RANGE` is ignored.

Both write the same on-disk shape:
`<LABEL_SOURCE>_<date_or_static>_.../ <LABEL_SOURCE>_full_size.tiff` plus a
`userdata.json` sidecar.


In [ ]:
from geoai_datacubes.fetch import fetch_sentinel_data

t0 = time.time()
lulc_data, lulc_bands = fetch_sentinel_data(
    LABEL_SOURCE,
    bands=["LULC"],                    # just the hard-classification band
    time_range=TIME_RANGE,             # ignored by ESA-WorldCover (static=True)
    roi=AOI,
    resolution=RESOLUTION,
    save_folder=str(DATA),
    provider="auto",                   # -> earth_engine or planetary_computer
)
lulc_scene = sorted(DATA.glob(f"{LABEL_SOURCE}_*"), key=os.path.getmtime)[-1]
print(f"\n{LABEL_SOURCE} fetched in {time.time()-t0:.1f}s")
print(f"scene folder : {lulc_scene.relative_to(REPO_ROOT)}")
print(f"bands        : {lulc_bands}")
print(f"array shape  : {lulc_data[0].shape}")


## Inspect and visualise the LULC map

Open the GeoTIFF that the provider wrote to disk, confirm the CRS + pixel
grid, and render the label band as a categorical map. Dynamic World uses the
9-class palette from Brown et al. 2022 (class IDs `0..8`); ESA-WorldCover uses
its own 11-class palette (class IDs `10, 20, ..., 100`). A tiny dispatcher
below hides that difference.


In [ ]:
# Journal-figure rcParams: bold labels, thick axes, no title.
mpl.rcParams.update({
    "font.size": 14, "axes.labelsize": 16,
    "xtick.labelsize": 13, "ytick.labelsize": 13, "legend.fontsize": 12,
    "font.weight": "bold", "axes.labelweight": "bold",
    "axes.linewidth": 1.6, "axes.edgecolor": "black",
    "xtick.major.width": 1.6, "ytick.major.width": 1.6,
    "xtick.major.size": 6, "ytick.major.size": 6,
    "xtick.color": "black", "ytick.color": "black",
    "xtick.direction": "in", "ytick.direction": "in",
    "lines.linewidth": 2.2, "lines.markersize": 7,
    "legend.frameon": True, "legend.edgecolor": "black",
    "savefig.dpi": 300, "savefig.bbox": "tight",
})

# Per-product palette: {class_id: (hex_colour, class_label)}.
PALETTES = {
    "Dynamic-World": {   # 0..8, Brown et al. 2022 official palette
        0: ("#419BDF", "water"),        1: ("#397D49", "trees"),
        2: ("#88B053", "grass"),        3: ("#7A87C6", "flooded veg"),
        4: ("#E49635", "crops"),        5: ("#DFC35A", "shrub/scrub"),
        6: ("#C4281B", "built"),        7: ("#A59B8F", "bare"),
        8: ("#B39FE1", "snow/ice"),
    },
    "ESA-WorldCover": {  # 10..100, ESA's canonical palette
        10:  ("#006400", "trees"),      20:  ("#FFBB22", "shrubland"),
        30:  ("#FFFF4C", "grassland"),  40:  ("#F096FF", "cropland"),
        50:  ("#FA0000", "built-up"),   60:  ("#B4B4B4", "bare/sparse"),
        70:  ("#F0F0F0", "snow/ice"),   80:  ("#0064C8", "water"),
        90:  ("#0096A0", "wetland"),    95:  ("#00CF75", "mangroves"),
        100: ("#FAE6A0", "moss/lichen"),
    },
}

lulc_tiff = lulc_scene / f"{LABEL_SOURCE}_full_size.tiff"
with rasterio.open(lulc_tiff) as src:
    lulc = src.read(1)
    print(f"path       : {lulc_tiff.relative_to(REPO_ROOT)}")
    print(f"CRS        : {src.crs}")
    print(f"shape      : {src.height} x {src.width} px")
    print(f"pixel size : {src.transform.a:.2f} x {abs(src.transform.e):.2f} m")

palette = PALETTES[LABEL_SOURCE]
class_ids = sorted(palette.keys())
colours   = [palette[c][0] for c in class_ids]
labels    = [palette[c][1] for c in class_ids]

# Remap the observed integer classes into 0..K-1 slots for a compact colour
# ramp; unmapped pixels (NaN or unknown codes) render as light grey.
lulc_int = np.where(np.isfinite(lulc), np.rint(lulc).astype(np.int64), -1)
remap    = np.full(lulc_int.shape, -1, dtype=np.int64)
for k, cid in enumerate(class_ids):
    remap[lulc_int == cid] = k

cmap = ListedColormap(["#DDDDDD"] + colours)
norm = BoundaryNorm(np.arange(-1.5, len(class_ids) + 0.5, 1), cmap.N)

fig, ax = plt.subplots(figsize=(8, 6.5))
im = ax.imshow(remap, cmap=cmap, norm=norm, interpolation="nearest")
ax.set_xlabel("x [px]")
ax.set_ylabel("y [px]")
cbar = fig.colorbar(im, ax=ax, ticks=np.arange(len(class_ids)), shrink=0.85)
cbar.ax.set_yticklabels(labels, fontweight="bold")
cbar.set_label(f"{LABEL_SOURCE} class", fontweight="bold")
plt.show()

# Class breakdown for context.
uniq, counts = np.unique(remap[remap >= 0], return_counts=True)
total = counts.sum()
print(f"\nClass breakdown over the AOI ({LABEL_SOURCE}):")
for k, c in zip(uniq, counts):
    print(f"  {labels[k]:<14s} {100*c/total:5.1f}%")


## Fetch Sentinel-2

Grab RGB + NIR (`B04`, `B03`, `B02`, `B08`) over the same AOI and time
window. `provider="auto"` routes S2 to `earthsearch` (AWS Element-84 STAC);
`max_cloud_coverage=0.10` picks the least-cloudy scene in the window.


In [ ]:
t0 = time.time()
s2_data, s2_bands = fetch_sentinel_data(
    "Sentinel-2",
    bands=["B04", "B03", "B02", "B08"],
    time_range=TIME_RANGE,
    roi=AOI,
    resolution=RESOLUTION,
    save_folder=str(DATA),
    max_cloud_coverage=0.10,
    provider="auto",                   # -> earthsearch
)
s2_scene = sorted(DATA.glob("Sentinel-2_*"), key=os.path.getmtime)[-1]
print(f"\nSentinel-2 fetched in {time.time()-t0:.1f}s")
print(f"scene folder : {s2_scene.relative_to(REPO_ROOT)}")
print(f"bands        : {s2_bands}")


## Fetch Copernicus DEM

The DEM is static so `TIME_RANGE` is ignored. Copernicus GLO-30 is native
30 m; asking for it at `RESOLUTION=10` means the fetcher will pull the 30 m
tile and the fusion step will upsample it bilinearly to the 10 m master
grid. Bilinear upsampling of a smooth elevation surface is fine as an ML
feature -- we are not doing hydrology here.


In [ ]:
t0 = time.time()
dem_data, dem_bands = fetch_sentinel_data(
    "Copernicus-DEM",
    bands=["DEM"],
    time_range=TIME_RANGE,             # ignored (static=True)
    roi=AOI,
    resolution=RESOLUTION,
    save_folder=str(DATA),
    provider="auto",                   # -> earthsearch
)
dem_scene = sorted(DATA.glob("Copernicus-DEM_*"), key=os.path.getmtime)[-1]
print(f"\nCopernicus-DEM fetched in {time.time()-t0:.1f}s")
print(f"scene folder : {dem_scene.relative_to(REPO_ROOT)}")
print(f"bands        : {dem_bands}")


## Fuse into a single datacube

`fuse_response_tiffs` reprojects all three sources onto a common UTM grid at
`resolution=RESOLUTION` and clips to the intersection footprint. Band-by-band
resampling follows `preprocessing.fusion._NEAREST_BANDS`: the categorical
`LULC` band uses nearest-neighbour (class codes survive intact); continuous
reflectance and elevation bands use bilinear. Fused band names are prefixed
with the source mission -- e.g. `Sentinel-2_B04`, `Copernicus-DEM_DEM`,
`Dynamic-World_LULC` -- which is what the downstream feature-selection code
keys on.


In [ ]:
from geoai_datacubes.preprocessing.fusion import fuse_response_tiffs

# Order matters only for CRS: first input's CRS becomes the target grid CRS
# unless `dst_crs=` is passed. Putting S2 first pins the target to its
# native UTM zone.
inputs = [
    str(s2_scene   / "Sentinel-2_full_size.tiff"),
    str(dem_scene  / "Copernicus-DEM_full_size.tiff"),
    str(lulc_scene / f"{LABEL_SOURCE}_full_size.tiff"),
]
cube_path = OUT / "cube.tif"

t0 = time.time()
cube_meta = fuse_response_tiffs(
    inputs=inputs,
    output_path=str(cube_path),
    resolution=RESOLUTION,
    bbox_mode="intersection",
)
print(f"\nfused in {time.time()-t0:.1f}s")

with rasterio.open(cube_path) as src:
    cube_bands = list(src.descriptions)
    cube_shape = (src.count, src.height, src.width)
    cube_crs   = src.crs
print(f"bands  : {cube_bands}")
print(f"shape  : {cube_shape}")
print(f"CRS    : {cube_crs}")


## Extract features + label from the cube

The fused cube carries all three sources side-by-side. Split it into:

- **feature stack** -- the 5 continuous bands (`Sentinel-2_B04`,
  `Sentinel-2_B03`, `Sentinel-2_B02`, `Sentinel-2_B08`,
  `Copernicus-DEM_DEM`), normalised per band via the pipeline's
  `apply_band_norm` recipes from `MISSION_PROFILES` (S2 linear/10000, DEM
  mean-subtract in metres).
- **binary label** -- `1` where the fused LULC band equals `TARGET_CLASS_ID`,
  else `0`.

Reshape to a `(H*W, 5)` design matrix and a `(H*W,)` label vector so any
tabular model can consume them. Track a `valid` mask that drops pixels with
NaN in any feature or the label (fusion writes `NaN` outside the intersection
footprint).


In [ ]:
from geoai_datacubes.preprocessing.band_ops import apply_band_norm, get_band_norm
from geoai_datacubes.fetch.missions import MISSION_PROFILES

FEATURE_BANDS = [
    "Sentinel-2_B04", "Sentinel-2_B03", "Sentinel-2_B02", "Sentinel-2_B08",
    "Copernicus-DEM_DEM",
]
LABEL_BAND    = f"{LABEL_SOURCE}_LULC"

with rasterio.open(cube_path) as src:
    all_bands = list(src.descriptions)
    feature_idx = [all_bands.index(b) + 1 for b in FEATURE_BANDS]  # 1-based
    label_idx   = all_bands.index(LABEL_BAND) + 1
    feature_stack = np.stack([src.read(i) for i in feature_idx]).astype(np.float32)
    label_raw     = src.read(label_idx)

H, W = feature_stack.shape[1:]

# Per-band normalisation via the mission-profile recipes (linear/10000 for
# S2 spectral, mean_subtract/1000 for DEM). apply_band_norm ignores NaNs.
for i, name in enumerate(FEATURE_BANDS):
    recipe = get_band_norm(name, mission_profiles=MISSION_PROFILES)
    feature_stack[i] = apply_band_norm(feature_stack[i], recipe)

label_bin = (np.rint(label_raw).astype(np.int64) == TARGET_CLASS_ID).astype(np.int64)
valid_2d  = np.isfinite(feature_stack).all(axis=0) & np.isfinite(label_raw)

X_all = feature_stack.reshape(len(FEATURE_BANDS), -1).T          # (H*W, 5)
y_all = label_bin.reshape(-1)                                    # (H*W,)
v_all = valid_2d.reshape(-1)

n_pos = int(y_all[v_all].sum())
n_neg = int(v_all.sum() - n_pos)
print(f"cube shape      : ({H}, {W})")
print(f"valid pixels    : {int(v_all.sum()):,} / {v_all.size:,} "
      f"({100*v_all.mean():.1f}%)")
print(f"positive (=1)   : {n_pos:>9,d}  ({100*n_pos/max(1, n_pos+n_neg):.2f}%)")
print(f"negative (=0)   : {n_neg:>9,d}")


## Spatial train / test split

Neighbouring pixels in an EO scene are correlated, so a naive random pixel
split leaks near-duplicates between train and test and inflates metrics. We
instead cut the AOI vertically: **left 80% of columns -> train, right 20%
-> test**. Any within-column autocorrelation stays inside one side of the
split.


In [ ]:
split_col = int(0.80 * W)

col_ix = np.arange(W)
train_col_mask_2d = np.broadcast_to(col_ix <  split_col, (H, W))
test_col_mask_2d  = np.broadcast_to(col_ix >= split_col, (H, W))

train_mask = (train_col_mask_2d.reshape(-1)) & v_all
test_mask  = (test_col_mask_2d.reshape(-1))  & v_all

X_train, y_train = X_all[train_mask], y_all[train_mask]
X_test,  y_test  = X_all[test_mask],  y_all[test_mask]

def _line(name, y):
    n_pos = int(y.sum()); n = int(y.size)
    print(f"{name:<7s}  n={n:>9,d}  {TARGET_NAME}={n_pos:>7,d} "
          f"({100*n_pos/max(1, n):.2f}%)")
print(f"split at col {split_col} / {W}")
_line("train", y_train)
_line("test",  y_test)


## Train XGBoost

XGBoost is a good default for a pixel-level tabular classifier: fast on CPU,
handles heterogeneous feature scales natively, and returns feature
importances for free. We handle the (typically severe) target-class
imbalance with `scale_pos_weight = n_neg / n_pos`, which reweights the
positive class loss so the model does not just predict "background
everywhere" to satisfy accuracy.


In [ ]:
import xgboost as xgb

n_pos_tr = int(y_train.sum())
n_neg_tr = int(y_train.size - n_pos_tr)
spw      = max(1.0, n_neg_tr / max(1, n_pos_tr))

clf = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    tree_method="hist",
    eval_metric="logloss",
    scale_pos_weight=spw,
    n_jobs=-1,
    random_state=0,
)
t0 = time.time()
clf.fit(X_train, y_train)
print(f"XGB trained in {time.time()-t0:.1f}s  (scale_pos_weight={spw:.2f})")


## Evaluate

Report accuracy, precision, recall, F1 on the held-out right-hand strip,
draw the confusion matrix, plot feature importances, and render a
three-panel comparison over the test half of the AOI: S2 RGB, ground-truth
binary label, and the model's prediction.


In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix)

y_pred = clf.predict(X_test)

acc = accuracy_score(y_test,  y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec  = recall_score(y_test,   y_pred, zero_division=0)
f1   = f1_score(y_test,       y_pred, zero_division=0)
cm   = confusion_matrix(y_test, y_pred, labels=[0, 1])

print(f"accuracy  : {acc:.4f}")
print(f"precision : {prec:.4f}")
print(f"recall    : {rec:.4f}")
print(f"F1        : {f1:.4f}")
print(f"confusion :\n{cm}")

# ---- confusion matrix + feature importance ----------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

ax = axes[0]
im = ax.imshow(cm, cmap="Blues")
for (i, j), v in np.ndenumerate(cm):
    ax.text(j, i, f"{v:,}", ha="center", va="center",
            color="white" if v > cm.max() / 2 else "black", fontweight="bold")
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(["rest", TARGET_NAME], fontweight="bold")
ax.set_yticklabels(["rest", TARGET_NAME], fontweight="bold")
ax.set_xlabel("predicted"); ax.set_ylabel("true")

ax = axes[1]
importances = clf.feature_importances_
order = np.argsort(importances)
ax.barh(np.arange(len(order)), importances[order],
        color="#0072B2", edgecolor="black")
ax.set_yticks(np.arange(len(order)))
ax.set_yticklabels([FEATURE_BANDS[i] for i in order], fontweight="bold")
ax.set_xlabel("XGBoost feature importance")

plt.tight_layout()
plt.show()

# ---- 3-panel spatial figure over the test strip -----------------------
# Rebuild spatial views over the right-hand test strip using the same
# feature_stack we read earlier (avoids re-opening the cube).
rgb_bands = ["Sentinel-2_B04", "Sentinel-2_B03", "Sentinel-2_B02"]
rgb_idx   = [FEATURE_BANDS.index(b) for b in rgb_bands]
rgb_full  = feature_stack[rgb_idx]                                   # (3, H, W), normalised [0, 1]
# Small percentile stretch for display -- normalised spectral bands sit
# near the low end so a raw [0, 1] display looks black.
lo, hi = np.nanpercentile(rgb_full, [2, 98])
rgb_disp = np.clip((rgb_full - lo) / max(1e-9, hi - lo), 0, 1)
rgb_disp = np.moveaxis(rgb_disp, 0, -1)                               # (H, W, 3)

rgb_test  = rgb_disp[:, split_col:]
gt_test   = label_bin[:, split_col:].astype(np.float32)
valid_ts  = valid_2d[:, split_col:]

# Reconstruct a spatial prediction map: -1 for invalid pixels, else class.
pred_full = np.full((H, W), -1, dtype=np.int64)
pred_full.reshape(-1)[test_mask] = y_pred
pred_test = pred_full[:, split_col:].astype(np.float32)
gt_test[  ~valid_ts] = np.nan
pred_test[~valid_ts] = np.nan

bin_cmap = ListedColormap(["#DDDDDD", "#C4281B"])   # rest / target
bin_norm = BoundaryNorm([-0.5, 0.5, 1.5], bin_cmap.N)

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
axes[0].imshow(rgb_test)
axes[0].set_xlabel("(a) S2 RGB (test strip)")
axes[1].imshow(gt_test,   cmap=bin_cmap, norm=bin_norm, interpolation="nearest")
axes[1].set_xlabel(f"(b) ground truth: {TARGET_NAME}")
axes[2].imshow(pred_test, cmap=bin_cmap, norm=bin_norm, interpolation="nearest")
axes[2].set_xlabel(f"(c) XGBoost prediction: {TARGET_NAME}")
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()


## Where to go next

- **Try the other `LABEL_SOURCE`.** Flip cell 4 to `"ESA-WorldCover"`
  (target class `50` = built-up) and re-run: fully static, no EE call,
  same code path.
- **Try a deeper model.** Swap the XGBoost in cell 20 for a small U-Net
  or CNN patch classifier -- the fused cube already has the shape a
  segmentation model wants: `(bands, H, W)` on a common CRS.
- **Expand the AOI.** The Earth Engine provider auto-tiles large requests
  and stitches the response; STAC providers auto-mosaic. Grow `AOI` to
  county or state scale without touching any other code.
- **Fuse in more modalities.** Add Sentinel-1 SAR (`"Sentinel-1"`, bands
  `["VV", "VH"]`) or HLS (`"HLS_S30"` / `"HLS_L30"`) as extra inputs to
  `fuse_response_tiffs` for a richer feature stack. See
  `notebooks/00_geoai_datacubes_tour.ipynb` for the full mission
  inventory.
